# 05 — Model Evaluation

Loads the latest trained artifact and inspects it: metrics, threshold rationale, and the standard evaluation plots.

In [ ]:
from bnpl_credit_risk.models.persistence import ArtifactBundle
from bnpl_credit_risk.models.registry import resolve_model_dir
from bnpl_credit_risk.settings import get_settings, load_config

settings = get_settings()
config = load_config()
models_dir = settings.resolve(settings.artifacts_dir) / 'models'
bundle = ArtifactBundle.load(resolve_model_dir(models_dir, 'latest'))
bundle.version, bundle.metrics['test']['roc_auc'], bundle.threshold

In [ ]:
from bnpl_credit_risk.data.loaders import BNPLDataLoader
from bnpl_credit_risk.data.splitting import get_splitter

df = BNPLDataLoader(settings, config.data).load_raw()
splitter = get_splitter(config.training.split.strategy, config.training.split,
                        id_column=config.data.id_column, date_column=config.data.date_column,
                        target_column=config.data.target_column)
_, test_df = splitter.split(df)
X_test = test_df.drop(columns=[config.data.target_column])
y_test = test_df[config.data.target_column]
y_prob = bundle.pipeline.predict_proba(X_test)[:, 1]

In [ ]:
from bnpl_credit_risk.visualization.evaluation import (
                        plot_confusion_matrix_and_roc,
                        plot_probability_distribution,
)

y_pred = (y_prob >= bundle.threshold['threshold']).astype(int)
_ = plot_confusion_matrix_and_roc(y_test.to_numpy(), y_pred, y_prob)

In [ ]:
_ = plot_probability_distribution(y_prob)

## Feature importance

In [ ]:
from bnpl_credit_risk.features.feature_names import get_feature_importances
from bnpl_credit_risk.visualization.evaluation import plot_feature_importance

importances = get_feature_importances(bundle.pipeline)
_ = plot_feature_importance(importances, model_name=config.model.algorithm.upper())

## Calibration

In [ ]:
from bnpl_credit_risk.visualization.calibration import plot_calibration_curve

_ = plot_calibration_curve(y_test.to_numpy(), y_prob, label=bundle.version)